# PandoraZX ZX benchmarks

In [ ]:
import csv
import math
import random
import re
import time
import warnings
from contextlib import asynccontextmanager
from dataclasses import dataclass
from pathlib import Path
from re import Pattern
from typing import Awaitable, Callable

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "pandora").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the PandoraZX project root")

PROJECT_ROOT = find_project_root()
CIRCUITS_ROOT = PROJECT_ROOT / "circuits"
OPTIMIZER_ROOT = CIRCUITS_ROOT / "optimizer"
GENERATED_CIRC_ROOT = CIRCUITS_ROOT / "generated"
CONFIG_PATH = PROJECT_ROOT / "default_config.json"
RESULTS_PATH = PROJECT_ROOT / "benchmarking" / "zx_benchmark_results.csv"
PLOT_DIR = PROJECT_ROOT / "benchmarking" / "zx_benchmark_plots"

print(f"Project root: {PROJECT_ROOT}")

### Circuit and backend helpers

In [ ]:
import numpy as np
import pyzx as zx
import quizx
import qiskit.qasm2
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import SwapGate, IQP, QFT, RGQFTMultiplier, VBERippleCarryAdder

from pandora.db.core import PandoraDB
from pandora.db.repository import GateRepository
from pandora.db.service import PandoraService


SWAP_INSTRUCTION = qiskit.qasm2.CustomInstruction(
    "swap", 0, 2, SwapGate, builtin=True
)

def resolve_before_circuit(relative_path: str) -> Path:
    optimizer_root = OPTIMIZER_ROOT.resolve()
    path = (optimizer_root / relative_path).resolve()
    if optimizer_root not in path.parents:
        raise ValueError(f"Circuit must be inside {optimizer_root}")
    if not path.name.endswith("_before"):
        raise ValueError(f"Expected an exact *_before file, got {path.name!r}")
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def load_pyzx_circuit(path: Path) -> zx.Circuit:
    return zx.Circuit.load(str(path)).to_basic_gates()

def pyzx_to_qiskit(circuit: zx.Circuit):
    return qiskit.qasm2.loads(
        circuit.to_qasm(), custom_instructions=[SWAP_INSTRUCTION]
    )

@asynccontextmanager
async def open_pandora_service():
    
    db = PandoraDB(str(CONFIG_PATH))
    await db.connect()
    try:
        service = PandoraService(db=db, repo=GateRepository(db))
        await service.build_pandora()
        yield service
    finally:
        await db.close()


def simplify_pyzx(graph, mode: str) -> None:
    if mode == "basic":
        zx.simplify.basic_simp(graph)
    elif mode == "full":
        zx.simplify.full_reduce(graph)
    else:
        raise ValueError(f"Unknown simplification mode: {mode}")


async def simplify_pandora(service: PandoraService, mode: str) -> None:
    if mode == "basic":
        await service.simplify_zx_basic()
    elif mode == "full":
        await service.simplify_zx_full()
    else:
        raise ValueError(f"Unknown simplification mode: {mode}")


def simplify_quizx(graph, mode: str) -> None:
    if quizx is None:
        raise RuntimeError("QuiZX is not installed")
    if mode == "basic" and hasattr(quizx, "basic_simp"):
        quizx.basic_simp(graph)
    elif mode == "basic":
        raise NotImplementedError("This QuiZX version does not provide basic_simp")
    elif mode == "full":
        quizx.full_simp(graph)
    else:
        raise ValueError(f"Unknown simplification mode: {mode}")


def quizx_supports(mode: str) -> bool:
    if quizx is None:
        return False
    return mode == "full" or (mode == "basic" and hasattr(quizx, "basic_simp"))


def graph_counts(graph) -> dict[str, int]:
    return {
        "edges": int(graph.num_edges()),
        "spiders": sum(
            vertex_type in (zx.VertexType.Z, zx.VertexType.X)
            for vertex_type in graph.types().values()
        ),
    }


async def pandora_graph_counts(service: PandoraService) -> dict[str, int]:
    async with service.db.acquire() as conn:
        row = await conn.fetchrow("""
            SELECT (SELECT count(*) FROM zx_edge) AS edges,
                   (SELECT count(*) FROM zx_vertex
                    WHERE type IN (zx_const_z_spider(), zx_const_x_spider())) AS spiders
        """)
    return {"edges": int(row["edges"]), "spiders": int(row["spiders"])}


def count_columns(before: dict, after: dict) -> dict:
    return {
        "edges_before": before["edges"], "edges_after": after["edges"],
        "spiders_before": before["spiders"], "spiders_after": after["spiders"],
    }

In [ ]:
def measured(call: Callable[[], object]) -> tuple[object, float]:
    start = time.perf_counter()
    value = call()
    return value, time.perf_counter() - start


async def measured_async(call: Callable[[], Awaitable[object]]) -> tuple[object, float]:
    start = time.perf_counter()
    value = await call()
    return value, time.perf_counter() - start


def benchmark_pyzx(family: str, name: str, circuit: zx.Circuit, mode: str, run: int) -> dict:
    graph, circuit_to_zx_s = measured(circuit.to_graph)
    before = graph_counts(graph)
    _, simplify_s = measured(lambda: simplify_pyzx(graph, mode))
    after = graph_counts(graph)
    return {
        "circuit": name, "family": family,
        "qubits": circuit.qubits, "gates": len(circuit.gates),
        "run": run, "mode": mode, "backend": "pyzx",
        "circuit_to_zx_s": circuit_to_zx_s,
        "simplify_s": simplify_s,
        **count_columns(before, after),
    }


def benchmark_quizx(family: str, name: str, circuit: zx.Circuit, mode: str, run: int) -> dict:
    graph, circuit_to_zx_s = measured(
        lambda: circuit.to_graph(backend="quizx-vec")
    )
    before = graph_counts(graph)
    _, simplify_s = measured(lambda: simplify_quizx(graph, mode))
    after = graph_counts(graph)
    return {
        "circuit": name, "family": family,
        "qubits": circuit.qubits, "gates": len(circuit.gates),
        "run": run, "mode": mode, "backend": "quizx",
        "circuit_to_zx_s": circuit_to_zx_s,
        "simplify_s": simplify_s,
        **count_columns(before, after),
    }


async def benchmark_pandora(family: str, name: str, circuit: zx.Circuit, mode: str, run: int) -> dict:
    async with open_pandora_service() as service:
        await service.build_circuit(pyzx_to_qiskit(circuit))
        _, circuit_to_zx_s = await measured_async(
            service.convert_linked_circuit_to_zx
        )
        before = await pandora_graph_counts(service)
        _, simplify_s = await measured_async(
            lambda: simplify_pandora(service, mode)
        )
        after = await pandora_graph_counts(service)

    return {
        "circuit": name, "family": family,
        "qubits": circuit.qubits, "gates": len(circuit.gates),
        "run": run, "mode": mode, "backend": "pandora",
        "circuit_to_zx_s": circuit_to_zx_s,
        "simplify_s": simplify_s,
        **count_columns(before, after),
        
    }

### Checkpoint and resume helpers

In [ ]:
COUNT_COLUMNS = ["edges_before", "edges_after", "spiders_before", "spiders_after"]
RESULT_COLUMNS = [
    "circuit", "family", "qubits", "gates", "run", "mode", "backend",
    "circuit_to_zx_s", "simplify_s", *COUNT_COLUMNS,
]



def append_result(row: dict) -> None:
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    needs_header = not RESULTS_PATH.exists() or RESULTS_PATH.stat().st_size == 0
    columns = RESULT_COLUMNS
    if not needs_header:
        with RESULTS_PATH.open(newline="", encoding="utf-8") as results_file:
            columns = next(csv.reader(results_file))
        if not set(RESULT_COLUMNS).issubset(columns):
            raise ValueError("Checkpoint is missing result columns; choose a new RESULTS_PATH")
    # Preserve the header of older checkpoints that also have status/error columns.
    with RESULTS_PATH.open("a", newline="", encoding="utf-8") as results_file:
        writer = csv.DictWriter(results_file, fieldnames=columns)
        if needs_header:
            writer.writeheader()
        writer.writerow({column: row.get(column, "ok" if column == "status" else "")
                         for column in columns})
        results_file.flush()


def load_checkpoint() -> pd.DataFrame:
    if not RESULTS_PATH.exists() or RESULTS_PATH.stat().st_size == 0:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    previous = pd.read_csv(RESULTS_PATH)
    if not set(RESULT_COLUMNS).issubset(previous.columns):
        raise ValueError("Checkpoint is missing result columns; choose a new RESULTS_PATH")
    return previous


def load_completed_keys(previous: pd.DataFrame | None = None) -> set:
    previous = load_checkpoint() if previous is None else previous
    if previous.empty:
        return set()
    valid = previous.copy()
    if "status" in valid:
        valid = valid.loc[valid["status"].isna() | (valid["status"] == "ok")]
    valid = valid.loc[
        pd.to_numeric(valid["simplify_s"], errors="coerce").notna()
        & pd.to_numeric(valid["circuit_to_zx_s"], errors="coerce").notna()
        & (pd.to_numeric(valid["run"], errors="coerce") >= 1)
    ]
    return {
        (str(row.circuit), str(row.family), str(row.mode), str(row.backend), int(row.run))
        for row in valid.itertuples()
    }


def successful_rows(frame: pd.DataFrame) -> pd.DataFrame:
    numeric_columns = ["gates", "circuit_to_zx_s", "simplify_s", *COUNT_COLUMNS]
    for column in numeric_columns:
        frame[column] = pd.to_numeric(
            frame[column] if column in frame else pd.Series(index=frame.index, dtype=float),
            errors="coerce",
        )
    return frame

### Plotting helpers

In [ ]:
BACKENDS = ("pandora", "pyzx", "quizx")
BACKEND_STYLE = {
    "pandora": {"color": "cadetblue", "marker": "o"},
    "pyzx": {"color": "darkorange", "marker": "D"},
    "quizx": {"color": "tab:blue", "marker": "s"},
}
BACKEND_LABEL = {"pandora": "PandoraZX", "pyzx": "PyZX", "quizx": "QuiZX"}

def family_slug(family: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(family)).strip("_")


def save_figure(fig, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {output_path}")
    return output_path


def timing_axes(ax, data):
    ax.set_xlabel("Number of gates")
    ax.set_ylabel("Time (seconds)")
    if (data["gates"] > 0).all():
        ax.set_xscale("log", base=2)
    if (data["mean"] > 0).all() and (data["minimum"] > 0).all():
        ax.set_yscale("log")
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)


def timing_data(frame, metric, mode=None):
    clean = successful_rows(frame)
    clean = clean.loc[clean["backend"].isin(BACKENDS)]
    if mode is not None:
        clean = clean.loc[clean["mode"] == mode]
    clean = clean.dropna(subset=["gates", metric])
    clean = clean.loc[(clean["gates"] >= 0) & (clean[metric] >= 0)]
    return clean.groupby(
        ["family", "circuit", "gates", "backend"], as_index=False
    )[metric].agg(mean="mean", minimum="min", maximum="max")


def plot_simplification_by_family(frame, mode, output_dir):
    grouped = timing_data(frame, "simplify_s", mode)
    if grouped.empty:
        warnings.warn(f"No successful {mode} simplification measurements to plot")
        return {}
    # Every family in this mode uses the same time range, including error bars.
    time_min = grouped["minimum"].min()
    time_max = grouped["maximum"].max()
    if time_min == time_max:
        time_min, time_max = ((time_min / 2, time_max * 2)
                              if time_min > 0 else (0, 1))
    paths = {}
    for family, family_data in grouped.groupby("family", sort=True):
        fig, ax = plt.subplots()
        for backend in BACKENDS:
            data = family_data.loc[family_data["backend"] == backend].sort_values(
                ["gates", "circuit"]
            )
            if data.empty:
                continue
            ax.errorbar(
                data["gates"], data["mean"],
                yerr=[data["mean"] - data["minimum"], data["maximum"] - data["mean"]],
                label=BACKEND_LABEL[backend], **BACKEND_STYLE[backend],
                linewidth=1.5, capsize=3,
            )
        timing_axes(ax, family_data)
        ax.set_yscale("log")
        ax.set_ylim(time_min, time_max)
        circuit_labels = (family_data[["circuit", "gates"]].drop_duplicates()
                            .sort_values(["gates", "circuit"])
                            .groupby("gates", sort=True)["circuit"].agg(" / ".join))
        ax.set_xticks(circuit_labels.index, circuit_labels.tolist(),
                        rotation=40, ha="right", fontsize=8)
        ax.set_xlabel("Circuit")
        gate_axis = ax.secondary_xaxis("top")
        gate_axis.set_xticks(circuit_labels.index, [f"{gates:,.0f}" for gates in circuit_labels.index],
                            rotation=40, ha="left", fontsize=8)
        gate_axis.set_xlabel("Number of gates")
        fig.tight_layout()
        paths[family] = save_figure(
            fig, output_dir / f"simplify_zx_{mode}_{family_slug(family)}.pdf"
        )
    return paths


def plot_conversion_all_circuits(frame, output_path):
    grouped = timing_data(frame, "circuit_to_zx_s")
    if grouped.empty:
        warnings.warn("No successful circuit-to-ZX measurements to plot")
        return None
    fig, ax = plt.subplots()
    for backend in BACKENDS:
        data = grouped.loc[grouped["backend"] == backend].sort_values(
            ["gates", "family", "circuit"]
        )
        if data.empty:
            continue
        ax.plot(data["gates"], data["mean"], label=BACKEND_LABEL[backend],
                **BACKEND_STYLE[backend], linewidth=1.5, alpha=0.75)
    timing_axes(ax, grouped)
    fig.tight_layout()
    return save_figure(fig, output_path)

def plot_conversion_percentage_by_family(frame, output_dir):
    """Plot mean per-run conversion / (conversion + full rewrite) percentages."""
    from matplotlib.ticker import PercentFormatter

    clean = successful_rows(frame.copy())
    if "status" in clean:
        clean = clean.loc[clean["status"].fillna("ok") == "ok"]
    clean = clean.loc[
        (clean["mode"] == "full") & clean["backend"].isin(BACKENDS)
    ].copy()
    timing_columns = ["gates", "circuit_to_zx_s", "simplify_s"]
    clean[timing_columns] = clean[timing_columns].replace(
        [float("inf"), -float("inf")], float("nan")
    )
    clean = clean.dropna(subset=timing_columns)
    clean = clean.loc[(clean[timing_columns] >= 0).all(axis=1)].copy()
    # Pair timings within each run; omit zero or non-finite total times.
    total_time = clean["circuit_to_zx_s"] + clean["simplify_s"]
    clean = clean.loc[(total_time > 0) & (total_time < float("inf"))].copy()
    clean["conversion_pct"] = 100 * (clean["circuit_to_zx_s"] / total_time.loc[clean.index])
    grouped = clean.groupby(
        ["family", "circuit", "gates", "backend"], as_index=False
    )["conversion_pct"].mean()
    if grouped.empty:
        warnings.warn("No paired full-mode conversion and rewrite timings to plot")
        return {}

    paths = {}
    for family, family_data in grouped.groupby("family", sort=True):
        fig, ax = plt.subplots()
        for backend in BACKENDS:
            data = family_data.loc[family_data["backend"] == backend].sort_values(
                ["gates", "circuit"]
            )
            if data.empty:
                continue
            ax.plot(
                data["gates"], data["conversion_pct"], label=BACKEND_LABEL[backend],
                **BACKEND_STYLE[backend], linewidth=1.5, alpha=0.75,
            )
        ax.set_xlabel("Number of gates")
        if (family_data["gates"] > 0).all():
            ax.set_xscale("log", base=2)
        ax.set_ylabel("Portion of circuit to ZX conversion (%)")
        ax.set_ylim(0, 100)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
        ax.grid(alpha=0.2)
        ax.legend(frameon=False)
        fig.tight_layout()
        paths[family] = save_figure(
            fig, output_dir / f"circuit_to_zx_conversion_pct_{family_slug(family)}.pdf"
        )
    return paths

def full_reduction_data(frame, metric):
    """Mean per-run percentage reductions from paired, positive-baseline counts."""
    clean = successful_rows(frame)
    before, after = f"{metric}_before", f"{metric}_after"
    clean = clean.loc[(clean["mode"] == "full") & clean["backend"].isin(BACKENDS)]
    clean = clean.dropna(subset=[before, after])
    # A zero baseline has no defined percentage reduction; retain negative reductions.
    clean = clean.loc[(clean[before] > 0) & (clean[after] >= 0)].copy()
    clean["reduction_pct"] = 100 * (clean[after] - clean[before]) / clean[before]
    return clean.groupby(
        ["family", "circuit", "backend"], as_index=False
    )["reduction_pct"].mean()


def plot_full_reductions_by_family(frame, output_dir):
    clean = successful_rows(frame)
    clean = clean.loc[(clean["mode"] == "full") & clean["backend"].isin(BACKENDS)]
    reductions = {metric: full_reduction_data(frame, metric) for metric in ("spiders", "edges")}
    paths = {}
    for family, family_data in clean.groupby("family", sort=True):
        circuit_order = (family_data[["circuit", "gates"]].drop_duplicates()
                         .sort_values(["gates", "circuit"])["circuit"].drop_duplicates().tolist())
        family_reductions = {
            metric: data.loc[data["family"] == family]
            for metric, data in reductions.items()
        }
        if all(data.empty for data in family_reductions.values()):
            warnings.warn(f"{family}: no paired counts with positive baselines for reduction plots")
            continue
        positions = list(range(len(circuit_order)))
        paths[family] = {}
        for metric in ("spiders", "edges"):
            data = family_reductions[metric]
            if data.empty:
                warnings.warn(f"{family}: no {metric} reduction data")
                continue
            fig, ax = plt.subplots()
            for backend in BACKENDS:
                backend_data = data.loc[data["backend"] == backend]
                if backend_data.empty:
                    continue
                values = backend_data.set_index("circuit")["reduction_pct"].reindex(circuit_order)
                ax.plot(positions, values, label=BACKEND_LABEL[backend],
                        **BACKEND_STYLE[backend], linewidth=1.5)
            ax.legend(frameon=False)
            # ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
            ax.set_ylabel(f"{metric.removesuffix('s').capitalize()} change (%)")
            ax.set_xlabel("Circuit")
            ax.set_xticks(positions, circuit_order, rotation=40, ha="right", fontsize=8)
            ax.grid(alpha=0.2)
            fig.tight_layout()
            paths[family][metric] = save_figure(
                fig, output_dir / f"full_simp_{metric}_reduction_pct_{family_slug(family)}.pdf"
            )
    return paths


## Benchmark circuit utils

In [ ]:
@dataclass
class Case:
    name: str
    family: str
    circuit_loader: Callable[[], zx.Circuit]
    size: int | None = None

CLIFFORD_BASIS = ["rx", "ry", "rz", "h", "s", "sdg", "x", "y", "z", "cx", "cz", "swap",  "t", "tdg"]

def case_from_path(relative_path: str, name: str | None = None,
                   family: str | None = None, size: int | None = None) -> Case:
    if not name:
        name = str(Path(relative_path).name.removesuffix("_before"))
    if not family:
        family = str(Path(relative_path).parent.as_posix())

    if size is None:
        sizes_in_name = re.findall(r"\d+", name)
        size = int(sizes_in_name[-1]) if sizes_in_name else None

    return Case(
        size=size,
        name=name,
        family=family,
        circuit_loader=lambda path=relative_path: load_pyzx_circuit(resolve_before_circuit(path))
    )

def case_from_qiskit(family: str, max_exponent :int, generator) -> list[Case]:
    return case_from_qiskit_with_size(family, [2**n for n in range(1, max_exponent + 1)], generator)

def load_generated_circuit(path: Path, size: int, generator) -> zx.Circuit:
    # Generate only when an eligible backend actually needs this size.
    if not path.exists():
        circuit = generator(size)
        transpiled = transpile(circuit, basis_gates=CLIFFORD_BASIS, optimization_level=0)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(qiskit.qasm2.dumps(transpiled), encoding="utf-8")
    return zx.Circuit.load(str(path)).to_basic_gates()


def case_from_qiskit_with_size(family: str, sizes: list[int], generator) -> list[Case]:
    print("Selecting " + family)
    cases = []
    for size in sizes:
        name = f"{family} {size}"
        qasm_path = GENERATED_CIRC_ROOT / family / f"{name}.qasm"
        cases.append(Case(
            name=name, family=family, size=size,
            circuit_loader=lambda path=qasm_path, size=size: load_generated_circuit(
                path, size, generator
            ),
        ))
    return cases


def case_from_glob(family: str, folder: str, pattern: Pattern) -> list[Case]:
    paths = sorted(
        path.relative_to(OPTIMIZER_ROOT).as_posix()
        for path in (OPTIMIZER_ROOT / folder).iterdir()
        if  path.is_file() and pattern.fullmatch(path.name)
    )
    return [case_from_path(p, family=family) for p in paths]


def measurement_seconds(row: dict) -> float | None:
    try:
        seconds = max(float(row["simplify_s"]), 3 * float(row["circuit_to_zx_s"]))
    except (KeyError, TypeError, ValueError):
        return None
    return seconds if math.isfinite(seconds) and seconds >= 0 else None


def normalize_time_limits(time_limits, backends):
    """Accept one shared number/None or a dictionary of per-backend limits."""
    limits = dict(time_limits) if isinstance(time_limits, dict) else {
        backend: time_limits for backend in backends
    }
    for backend, limit in limits.items():
        if limit is None:
            continue
        try:
            valid = math.isfinite(limit) and limit > 0
        except TypeError:
            valid = False
        if not valid:
            raise ValueError(f"Time limit for {backend} must be positive or None")
    return limits


def record_time_cutoff(row, case_index, time_limits, cutoffs):
    limit = time_limits.get(row["backend"])
    if limit is None:
        return False
    if row.get("status") == "timeout":
        # A larger limit permits retrying a previously timed-out circuit.
        timeout_limit = row.get("time_limit_s")
        if timeout_limit is None or limit > float(timeout_limit):
            return False
    else:
        seconds = measurement_seconds(row)
        if seconds is None or seconds <= limit:
            return False
    key = (row["mode"], row["family"], row["backend"])
    cutoffs[key] = min(case_index, cutoffs.get(key, case_index))
    return True


def prepare_benchmark_plan(cases_by_mode, backends_by_mode, runs, time_limits, previous):
    if not isinstance(runs, int) or runs < 1:
        raise ValueError("RUNS must be a positive integer")
    time_limits = normalize_time_limits(time_limits, {
        backend for backends in backends_by_mode.values() for backend in backends
    })
    plan, case_indices = [], {}
    for mode, cases in cases_by_mode.items():
        families = {}
        for case in cases:
            families.setdefault(case.family, []).append(case)
        for family, family_cases in families.items():
            # Explicit sizes take precedence; unsized cases retain their list order.
            ordered = sorted(enumerate(family_cases), key=lambda item: (
                item[1].size is None,
                item[1].size if item[1].size is not None else item[0],
            ))
            for case_index, (_, case) in enumerate(ordered):
                identity = (mode, family, case.name)
                if identity in case_indices:
                    raise ValueError(f"Duplicate circuit in {mode}: {family} / {case.name}")
                case_indices[identity] = case_index
                for run in range(1, runs + 1):
                    for backend in backends_by_mode[mode]:
                        plan.append((mode, case, backend, run, case_index))

    completed = load_completed_keys(previous)
    cutoffs = {}
    for row in previous.to_dict("records"):
        try:
            key = (str(row["circuit"]), str(row["family"]), str(row["mode"]),
                   str(row["backend"]), int(row["run"]))
        except (TypeError, ValueError):
            continue
        identity = (key[2], key[1], key[0])
        if key in completed and identity in case_indices:
            record_time_cutoff(row, case_indices[identity], time_limits, cutoffs)
    return plan, completed, cutoffs


async def run_benchmark_plan(plan, completed, cutoffs, *, time_limits, measure, persist):
    time_limits = normalize_time_limits(time_limits, {entry[2] for entry in plan})
    counts = {"measured": 0, "already_complete": 0, "skipped_time_limit": 0, "failed": 0}
    for attempt, (mode, case, backend, run, case_index) in enumerate(plan, start=1):
        key = (case.name, case.family, mode, backend, run)
        if key in completed:
            counts["already_complete"] += 1
            continue
        cutoff_key = (mode, case.family, backend)
        if cutoff_key in cutoffs and case_index >= cutoffs[cutoff_key]:
            counts["skipped_time_limit"] += 1
            continue
        label = f"[{attempt:>4}/{len(plan)}] {backend:7} {mode:5} run={run} {case.name}"
        print(label)
        try:
            row = await measure(case, mode, backend, run)
        except Exception as exc:
            counts["failed"] += 1
            warnings.warn(f"{label} failed: {exc}")
            continue
        # Stop the pass if persistence fails, rather than lose expensive measurements.
        persist(row)
        completed.add(key)
        counts["measured"] += 1
        if record_time_cutoff(row, case_index, time_limits, cutoffs):
            reason = (f"timed out at {time_limits[backend]:g}s"
                      if row.get("status") == "timeout" else
                      f"time limit={measurement_seconds(row):.3f}s exceeded {time_limits[backend]:g}s")
            print(f"  {reason}; skipping remaining repeats and larger sizes "
                  f"for {backend} / {mode} / {case.family}")
    print(f"Benchmark pass: {counts}")
    return counts

In [ ]:
def iqp(n: int) -> QuantumCircuit:
    return IQP(np.ones((n, n), dtype=int))

## Benchmark configuration

In [ ]:
RUNS = 1
BACKEND_TIME_LIMIT_S = 90.0

In [ ]:
CASES_PER_MODE = {
    "basic" :[],
    "full":[]
}

MAX_EXPONENT = 12
EXP_2_SIZES = sorted([2**i for i in range(1, MAX_EXPONENT + 1)] + [3 * 2**i for i in range(MAX_EXPONENT)])


CASES_PER_MODE["full"].extend(case_from_glob("Adder_before", "QFT_and_Adders", re.compile(r"Adder\d+_before")))
CASES_PER_MODE["full"].extend(case_from_glob("gf2^En Mult", "Arithmetic_and_Toffoli", re.compile(r"gf2\^E\d+_mult_before")))
CASES_PER_MODE["full"].extend(case_from_glob("QFT_before", "QFT_and_Adders", re.compile(r"QFT\d+_before")))

CASES_PER_MODE["full"].extend(case_from_qiskit_with_size("QFT",                         EXP_2_SIZES, QFT))
CASES_PER_MODE["full"].extend(case_from_qiskit_with_size("VBE Adder",                   EXP_2_SIZES, VBERippleCarryAdder))
CASES_PER_MODE["full"].extend(case_from_qiskit_with_size("RGQFT",                       EXP_2_SIZES, RGQFTMultiplier))

CASES_PER_MODE["full"].extend(case_from_qiskit_with_size("IQP", EXP_2_SIZES, iqp))

In [ ]:
CASES_PER_MODE["basic"] = CASES_PER_MODE["full"]

In [ ]:
print(f"Selected {sum(len(cases) for cases in CASES_PER_MODE.values())} circuit/mode cases")
print(f"Results will be appended to {RESULTS_PATH}")

## Run or resume benchmarks

In [ ]:
backends_by_mode = {
    mode: ["pandora", "pyzx"] for mode in CASES_PER_MODE
}
for mode in CASES_PER_MODE:
    if quizx_supports(mode):
        backends_by_mode[mode].append("quizx")

previous = load_checkpoint()
benchmark_plan, completed, time_cutoffs = prepare_benchmark_plan(
    CASES_PER_MODE, backends_by_mode, RUNS,
    BACKEND_TIME_LIMIT_S, previous,
)
total = len(benchmark_plan)
already_done = sum(
    (case.name, case.family, mode, backend, run) in completed
    for mode, case, backend, run, _ in benchmark_plan
)
known_skips = sum(
    (case.name, case.family, mode, backend, run) not in completed
    and (mode, case.family, backend) in time_cutoffs
    and index >= time_cutoffs[(mode, case.family, backend)]
    for mode, case, backend, run, index in benchmark_plan
)
print(f"Planned measurements before time cutoffs: {total}")
print(f"Already complete: {already_done}")
print(f"Skipped by saved time cutoffs: {known_skips}")
print(f"Remaining eligible measurements: {total - already_done - known_skips}")
print(f"Backends by mode: {backends_by_mode}")
print(f"Time limit: {BACKEND_TIME_LIMIT_S}")

In [ ]:
async def measure_case(case: Case, mode: str, backend: str, run: int) -> dict:
    circuit = case.circuit_loader()
    if backend == "pyzx":
        return benchmark_pyzx(case.family, case.name, circuit, mode, run)
    if backend == "quizx":
        return benchmark_quizx(case.family, case.name, circuit, mode, run)
    if backend == "pandora":
        return await benchmark_pandora(case.family, case.name, circuit, mode, run)
    raise ValueError(f"Unknown backend: {backend}")


benchmark_pass = await run_benchmark_plan(
    benchmark_plan, completed, time_cutoffs,
    time_limits=BACKEND_TIME_LIMIT_S,
    measure=measure_case, persist=append_result,
)
print(f"Results: {RESULTS_PATH}")

## Load and summarize results

In [ ]:
results = pd.read_csv(RESULTS_PATH)

display(
    results.groupby(["mode", "backend"])
    .size()
    .rename("rows")
    .reset_index()
)

display(
    results.groupby(["family"])
    .agg(
        runs=("run", "count"),
        simplify_min=("simplify_s", "min"),
        simplify_max=("simplify_s", "max"),
    )
)

display(results.loc[
    results.groupby(["backend", "mode", "family"])["simplify_s"].idxmax(),
    ["family", "backend", "mode", "circuit", "simplify_s"]
].sort_values(["family", "mode", "backend"]))

## Figures

In [ ]:
plt.style.use(PROJECT_ROOT / "benchmarking/dippa.mplstyle")

In [ ]:
conversion_plot = plot_conversion_all_circuits(
    results, PLOT_DIR / "circuit_to_zx_conversion.pdf"
)

In [ ]:
conversion_percentage_plots = plot_conversion_percentage_by_family(results, PLOT_DIR)

In [ ]:
plot_simplification_by_family(results, "basic", PLOT_DIR)

In [ ]:
plot_simplification_by_family(results, "full", PLOT_DIR)

In [ ]:
reduction_plots = plot_full_reductions_by_family(results, PLOT_DIR)

## Profile one PandoraZX circuit

In [ ]:
async def profile_pandora_rewrites(circuit: zx.Circuit, mode: str = "full", timeout: float | None = None) -> dict:
    """Return rewritten-spider throughput and per-method statistics for one circuit.

    Count the input spiders of each applied rewrite, including repeated uses
    of the same spider. Divide by simplification wall time, excluding loading
    and conversion. Procedure times may overlap, so do not sum them for this rate.
    """
    if mode not in ("basic", "full"):
        raise ValueError(f"Unknown simplification mode: {mode}")

    async with open_pandora_service() as service:
        await service.build_circuit(pyzx_to_qiskit(circuit))
        await service.convert_linked_circuit_to_zx()
        _, simplify_s = await measured_async(
            lambda: simplify_pandora(service, mode)
        )

        async with service.db.acquire() as conn:
            rows = await conn.fetch("""
                SELECT procedure_name,
                       count(*) AS call_count,
                       sum(execution_count)::bigint AS rewrite_count,
                       sum(elapsed_seconds) * 1000 AS total_ms,
                       avg(elapsed_seconds) * 1000 AS avg_ms,
                       min(elapsed_seconds) * 1000 AS min_ms,
                       max(elapsed_seconds) * 1000 AS max_ms
                FROM zx_rewrite_runs
                GROUP BY procedure_name
                ORDER BY procedure_name
            """)

    results = [dict(row) for row in rows]
    print(f"PandoraZX {mode} rewrite profile: {circuit.qubits} qubits, {len(circuit.gates)} gates")
    print(f"{'Procedure':32} {'Calls':>8} {'Iterations':>10} {'Avg (ms)':>12} {'Min (ms)':>12} {'Max (ms)':>12}")
    for row in results:
        print(f"{row['procedure_name']:32} {row['call_count']:8d} {row['rewrite_count']:10d} {row['avg_ms']:12.3f} {row['min_ms']:12.3f} {row['max_ms']:12.3f}")



In [ ]:
await profile_pandora_rewrites(case_from_qiskit("IQP", 4, iqp)[1].circuit_loader())